In [1]:
train_data = [
    {"text": "Q: What is AI?\nA: Artificial Intelligence is the simulation of human intelligence."},
    {"text": "Q: What is Python?\nA: Python is a programming language."},
    {"text": "Q: What is ML?\nA: Machine Learning is a subset of AI."},
]

In [2]:
!pip install transformers datasets torch

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset

In [4]:
data = [
    {"text": "Instruction: What is Artificial Intelligence?\nResponse: Artificial Intelligence is the simulation of human intelligence by machines."},
    {"text": "Instruction: Explain Machine Learning\nResponse: Machine Learning is a subset of AI that allows systems to learn from data."},
    {"text": "Instruction: What is Python used for?\nResponse: Python is used for web development, data science, AI, and automation."},
    {"text": "Instruction: What is Deep Learning?\nResponse: Deep Learning is a subset of machine learning that uses neural networks with many layers."},
    {"text": "Instruction: What is data science?\nResponse: Data science involves extracting insights from structured and unstructured data."},
    {"text": "Instruction: What is cloud computing?\nResponse: Cloud computing provides on-demand computing resources over the internet."},
    {"text": "Instruction: What is NLP?\nResponse: Natural Language Processing enables machines to understand and process human language."},
    {"text": "Instruction: What is supervised learning?\nResponse: Supervised learning uses labeled data to train models."},
    {"text": "Instruction: What is unsupervised learning?\nResponse: Unsupervised learning finds patterns in unlabeled data."},
    {"text": "Instruction: What is reinforcement learning?\nResponse: Reinforcement learning trains agents using rewards and penalties."}
]

dataset = Dataset.from_list(data)

In [5]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_steps=1,
    save_steps=10,
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [9]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,7.922724
2,5.635256
3,3.244104
4,1.984632
5,1.698428
6,1.540488
7,1.476835
8,1.383921
9,1.400580
10,1.300740


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25, training_loss=1.7047703266143799, metrics={'train_runtime': 89.3977, 'train_samples_per_second': 0.559, 'train_steps_per_second': 0.28, 'total_flos': 816552345600.0, 'train_loss': 1.7047703266143799, 'epoch': 5.0})

In [11]:
def test_model(query):
    input_text = f"Instruction: {query}\nResponse:"

    inputs = tokenizer(input_text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(output)

In [12]:
test_model("What is AI?")
test_model("Explain cloud computing")
test_model("What is reinforcement learning?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: What is AI?
Response: AI is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is a computer that is


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Explain cloud computing
Response: What is cloud computing?
Instruction: What is reinforcement learning?
Response: Learning is a process that involves learning from different sources.
